# 03 – Baseline LightGBM

Premier modèle de prédiction de la RUL. Il servira de référence pour le LSTM (étape 4) et de base pour la conformal prediction (étape 5).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.data.load import PROJECT_ROOT
from src.features.preprocess import FEATURES_FD001
from src.features.rolling import add_rolling_features, last_cycle
from src.models.metrics import rmse, nasa_score

sns.set_theme(style="whitegrid")
PROC = PROJECT_ROOT / "data" / "processed"

## 1. Chargement et features glissantes

Les capteurs sont très bruités. Pour chaque capteur on ajoute, sur des fenêtres de 5, 10 et 20 cycles passés : la moyenne (lissage), l'écart-type (instabilité) et la pente (vitesse de dégradation). On ajoute aussi le cycle courant (l'âge du moteur).

In [ ]:
sets = {k: pd.read_csv(PROC / f"FD001_{k}.csv") for k in ["train", "calib", "val", "test"]}
sets = {k: add_rolling_features(v, FEATURES_FD001, windows=(5, 10, 20)) for k, v in sets.items()}

FEATS = [c for c in sets["train"].columns if c not in ("unit", "RUL")]
print(f"{len(FEATS)} features")
X = {k: v[FEATS] for k, v in sets.items()}
y = {k: v["RUL"] for k, v in sets.items()}

## 2. Entraînement

Entraînement sur le train, early stopping sur la validation : on arrête d'ajouter des arbres quand l'erreur de validation ne s'améliore plus.

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.7,
    reg_lambda=1.0,
    random_state=42,
    verbose=-1,
)
model.fit(
    X["train"], y["train"],
    eval_set=[(X["val"], y["val"])],
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)
print("Meilleur nombre d'arbres :", model.best_iteration_)

## 3. Évaluation sur la validation

Deux protocoles :
- **toutes les lignes** : erreur moyenne sur toute la vie des moteurs ;
- **dernier cycle** : le protocole officiel du challenge (une prédiction par moteur).

In [ ]:
def evaluate(name):
    df = sets[name].copy()
    df["pred"] = np.clip(model.predict(X[name]), 0, None)
    last = last_cycle(df)
    return {
        "RMSE (toutes lignes)": rmse(df.RUL, df.pred),
        "RMSE (dernier cycle)": rmse(last.RUL, last.pred),
        "Score NASA (dernier cycle)": nasa_score(last.RUL, last.pred),
        "n moteurs": df.unit.nunique(),
    }, df

res_val, val_pred = evaluate("val")
pd.Series(res_val).round(2)

## 4. Visualisation des prédictions

Pour quelques moteurs de validation : RUL prédite vs vraie RUL au fil des cycles.

In [ ]:
units = val_pred.unit.unique()[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharey=True)
for ax, u in zip(axes.flat, units):
    d = val_pred[val_pred.unit == u]
    ax.plot(d.cycle, d.RUL, "k--", label="Vraie RUL")
    ax.plot(d.cycle, d.pred, color="steelblue", label="Prédiction")
    ax.set(title=f"Moteur {u}", xlabel="Cycle")
axes[0, 0].set_ylabel("RUL")
axes[0, 0].legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(val_pred.RUL, val_pred.pred, s=4, alpha=0.3)
ax.plot([0, 125], [0, 125], "r--")
ax.set(xlabel="Vraie RUL", ylabel="RUL prédite", title="Validation : prédiction vs vérité")
plt.show()

## 5. Erreur selon la RUL

Le modèle est-il plus précis près de la panne ? C'est là que la précision compte le plus pour la maintenance.

In [ ]:
val_pred["err"] = val_pred.pred - val_pred.RUL
val_pred["tranche"] = pd.cut(val_pred.RUL, bins=[-1, 25, 50, 75, 100, 125], labels=["0-25", "25-50", "50-75", "75-100", "100-125"])
print(val_pred.groupby("tranche", observed=True)["err"].agg(rmse=lambda e: np.sqrt((e**2).mean()), biais="mean").round(2))

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=val_pred, x="tranche", y="err", ax=ax, color="steelblue")
ax.axhline(0, color="red", ls="--")
ax.set(xlabel="Vraie RUL", ylabel="Erreur (prédit - vrai)", title="Distribution de l'erreur par tranche de RUL")
plt.show()

## 6. Importance des features

In [ ]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=FEATS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 6))
(imp.head(20) / imp.sum()).iloc[::-1].plot.barh(ax=ax, color="steelblue")
ax.set(title="Top 20 features (gain relatif)", xlabel="Part du gain total")
plt.show()

## 7. Évaluation finale sur le test officiel

⚠️ On ne regarde ce résultat **qu'une fois**, pour pouvoir se comparer à la littérature. On ne modifie pas le modèle en fonction de ce chiffre, sinon le test devient un deuxième jeu de validation.

In [ ]:
res_test, test_pred = evaluate("test")
pd.DataFrame({"validation": res_val, "test": res_test}).round(2)

## 8. Sauvegarde du modèle

In [ ]:
model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(exist_ok=True)
joblib.dump({"model": model, "features": FEATS}, model_dir / "lgbm_fd001.joblib")
print("Modèle sauvegardé")

## 9. Conclusions

_À compléter :_
- RMSE validation / test :
- Le modèle est-il meilleur près de la panne ou en début de vie ?
- Biais : prédit-il plutôt trop tôt ou trop tard ?
- Features les plus importantes :